# GUS04B — Estimator Setup & Skeleton Validation

**Work chunk B (v5.1) — Architecture & Infrastructure**

This notebook demonstrates that the `DemographicEstimator` skeleton works
correctly with the real database. We:

1. Load the database and import the estimator module
2. Verify constants, ANCHOR_SUBJECTS, and E_SUBJECT_NAMES
3. Test `_get_aggregation_children()` on sample records
4. Test `_store_estimated_cross_table()` with dummy data
5. Verify provenance queries
6. Confirm E_ subjects appear correctly in the DB
7. Test the DB integration wrapper methods

All estimation pipeline stubs raise `NotImplementedError` as expected —
actual numerical implementation comes in work chunks C–D.

In [14]:
import sys, os, warnings
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings('ignore')

# Paths
DB_PATH = Path(os.path.expanduser(
    '~/Documents/Studium Volkswirschaftslehre/3. Semester/'
    'Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_O.pkl'
))
TOOLS_PATH = DB_PATH.parents[2] / 'local_repo' / 'LRDWI-Paper' / 'Code' / 'tools'
sys.path.insert(0, str(TOOLS_PATH))

from geoTERYT_db import (
    load_complete_database, GeoTERYTDatabase, TERYTRecord,
    DataSeries, CrossTable, YEAR_RANGE_FULL, DATETIME_INDEX_FULL,
    LEVEL_VOIVODESHIP, LEVEL_POWIAT, LEVEL_GMINA,
    RODZ_SUB_DIVISIONS,
)
from demographic_estimator import (
    DemographicEstimator, ProvenanceMask,
    ANCHOR_SUBJECTS, E_SUBJECT_NAMES,
    PREDICTION_1990_RANGE, PREDICTION_2000_RANGE,
    CENSUS_YEARS, RODZ_AGGREGATION_SET,
    _get_aggregation_children,
    EPSILON, IPF_MAX_ITER, IPF_CONVERGENCE,
    GUROBI_AVAILABLE, IPFN_AVAILABLE,
)

print('Imports OK')

Imports OK


In [3]:
# ── Step 1: Load database ──
db = load_complete_database(DB_PATH, verbose=True)
records = db._records
print(f'\nRecords: {len(records):,}')

Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_O.pkl...
  Database version: 4.3
  ✓ Restored old voivodships: 49 rows
  ✓ Restored geometry store: 13,287 unique geometries
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4612 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 3661
  ✓ Records with old_woj: 4104
  ✓ Records with data: 4584
  ✓ Records with cross tables: 4584
  ✓ Records with population data: 4582
  ✓ Records with pop_class: 3411

Records: 4,612


## 1. Constants & Configuration

In [4]:
# ── Step 2: Verify constants ──
print('EPSILON          :', EPSILON)
print('IPF_MAX_ITER     :', IPF_MAX_ITER)
print('IPF_CONVERGENCE  :', IPF_CONVERGENCE)
print('GUROBI_AVAILABLE :', GUROBI_AVAILABLE)
print('IPFN_AVAILABLE   :', IPFN_AVAILABLE)
print()
print('PREDICTION_1990_RANGE:', list(PREDICTION_1990_RANGE))
print('PREDICTION_2000_RANGE:', list(PREDICTION_2000_RANGE))
print('CENSUS_YEARS         :', CENSUS_YEARS)
print('RODZ_AGGREGATION_SET :', RODZ_AGGREGATION_SET)
print()
print('ANCHOR_SUBJECTS:')
for var_type, sections in ANCHOR_SUBJECTS.items():
    for section, cfg in sections.items():
        print(f'  {var_type:12s} / {section}: anchors={cfg["anchor_subjects"]}, '
              f'marginals={cfg["marginal_subjects"]}')
print()
print('E_SUBJECT_NAMES:')
for key, name in E_SUBJECT_NAMES.items():
    print(f'  {str(key):30s} → {name}')

EPSILON          : 1e-10
IPF_MAX_ITER     : 1000
IPF_CONVERGENCE  : 1e-06
GUROBI_AVAILABLE : True
IPFN_AVAILABLE   : True

PREDICTION_1990_RANGE: [1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002]
PREDICTION_2000_RANGE: [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
CENSUS_YEARS         : [1988, 2002, 2011, 2021]
RODZ_AGGREGATION_SET : {'2', '1', '3'}

ANCHOR_SUBJECTS:
  age_sex      / 2000: anchors=['M_age_sex'], marginals=['M_age_sex']
  age_sex      / 1990: anchors=['M_age_sex', 'M_age_1990'], marginals=['H_age_sex']
  educ         / 2000: anchors=['M_educ_2000'], marginals=['P2350', 'P4092']
  educ         / 1990: anchors=['M_educ_1990'], marginals=['H_sex_educ']
  educ_sex     / 2000: anchors=['M_educ_sex_2000'], marginals=['P2350', 'P4092']
  educ_sex     / 1990: anchors=['M_educ_sex_1990'], marginals=['H_sex_educ'

## 2. Estimator Initialization

In [5]:
# ── Step 3: Initialize DemographicEstimator ──
est = DemographicEstimator(db, verbose=True)
print()
print(repr(est))

DemographicEstimator initialised  (Gurobi=YES, IPFN=YES)

DemographicEstimator(completed=0/9, Gurobi=YES)


## 3. Aggregation Children — `_get_aggregation_children()`

In [6]:
# ── Step 4: Test _get_aggregation_children() ──
# Test at various levels: country, voivodeship, powiat, and gmina

test_cases = [
    ('0000000', 'Country'),
    ('0200000', 'Voivodeship (Dolnośląskie)'),
    ('1400000', 'Voivodeship (Mazowieckie)'),
    ('0201000', 'Powiat (Bolesławiecki)'),
    ('1465000', 'Powiat (m.st. Warszawa)'),
    ('0201011', 'Gmina (Bolesławiec — urban)'),
]

for tid, label in test_cases:
    rec = records.get(tid)
    if rec is None:
        print(f'{tid} ({label}): NOT FOUND')
        continue
    children = _get_aggregation_children(rec, db)
    print(f'{tid} ({label}): {len(children)} children')
    if len(children) <= 15:
        for c in children:
            cr = records.get(c)
            cname = cr.name if cr else '???'
            print(f'    {c}  {cname}')
    else:
        print(f'    (first 5: {", ".join(children[:5])} ...)')

0000000 (Country): 16 children
    (first 5: 0200000, 0400000, 0600000, 0800000, 1000000 ...)
0200000 (Voivodeship (Dolnośląskie)): 30 children
    (first 5: 0201000, 0202000, 0203000, 0204000, 0205000 ...)
1400000 (Voivodeship (Mazowieckie)): 43 children
    (first 5: 1401000, 1402000, 1403000, 1404000, 1405000 ...)
0201000 (Powiat (Bolesławiecki)): 6 children
    0201011  Bolesławiec
    0201022  Bolesławiec
    0201032  Gromadka
    0201043  Nowogrodziec
    0201052  Osiecznica
    0201062  Warta Bolesławiecka
1465000 (Powiat (m.st. Warszawa)): 1 children
    1465011  Warszawa
0201011 (Gmina (Bolesławiec — urban)): 0 children


### 3a. Validate: country ↔ voivodeships consistency

In [7]:
# ── Step 5: Validate country = Σ voivodeships for population ──
country_rec = records['0000000']
voivs = _get_aggregation_children(country_rec, db)
print(f'Number of voivodeships from _get_aggregation_children: {len(voivs)}')
print(f'Expected: 16')
print()

# Check for a census year where pop is available
for yr in [2002, 2005, 2011, 2021]:
    ts = pd.Timestamp(yr, 1, 1)
    country_pop = country_rec.pop.get(ts, np.nan)
    if pd.notna(country_pop):
        voiv_sum = sum(records[v].pop.get(ts, 0) for v in voivs)
        diff = country_pop - voiv_sum
        print(f'{yr}: Country pop = {country_pop:,.0f}, '
              f'Σ voiv = {voiv_sum:,.0f}, diff = {diff:,.0f}')

Number of voivodeships from _get_aggregation_children: 16
Expected: 16

2002: Country pop = 38,218,531, Σ voiv = 38,218,531, diff = 0
2005: Country pop = 38,157,055, Σ voiv = 38,157,055, diff = 0
2011: Country pop = 38,538,447, Σ voiv = 38,538,447, diff = 0
2021: Country pop = 37,907,704, Σ voiv = 37,907,704, diff = 0


## 4. Store Estimated Cross Table — Dummy Data Test

In [8]:
# ── Step 6: Test _store_estimated_cross_table() with synthetic data ──
# Pick a gmina and create a small dummy 1D E_ cross table

TEST_TID = '0201011'   # Bolesławiec — urban gmina
TEST_E_SID = 'E_hh_size_2000'  # 1D subject: household sizes
TEST_YEAR = 2005

dim_names = ['n1']
dim_labels = {'n1': ['1 osoba', '2 osoby', '3 osoby', '4 osoby', '5 osób', '6 i więcej osób']}

# Create fake distribution
dummy_table = np.array([1200.0, 2400.0, 1800.0, 1500.0, 900.0, 600.0])

est._store_estimated_cross_table(
    teryt_id=TEST_TID,
    e_subject_id=TEST_E_SID,
    year=TEST_YEAR,
    table=dummy_table,
    dim_names=dim_names,
    dim_labels=dim_labels,
    is_observed=False,
)

# Verify it shows up on the record
rec = records[TEST_TID]
ct = rec.cross_tables.get(TEST_E_SID)
print('CrossTable stored:', ct is not None)
print(f'  dim_names : {ct.dim_names}')
print(f'  dim_labels: {ct.dim_labels}')
print(f'  shape     : {ct._shape}')
print(f'  table[{TEST_YEAR}] : {ct.tables[TEST_YEAR]}')
print(f'  sum       : {ct.tables[TEST_YEAR].sum():.0f}')
print()

# Verify DataSeries counterparts were created
e_keys = [k for k in rec.data if k[0] == 'Estimated' and k[1] == TEST_E_SID]
print(f'DataSeries entries for {TEST_E_SID}: {len(e_keys)}')
for k in e_keys:
    ds = rec.data[k]
    val = ds.values.get(pd.Timestamp(TEST_YEAR, 1, 1), np.nan)
    print(f'  {k[2]}: categories={ds.categories}, value={val}')

CrossTable stored: True
  dim_names : ['n1']
  dim_labels: {'n1': ['1 osoba', '2 osoby', '3 osoby', '4 osoby', '5 osób', '6 i więcej osób']}
  shape     : (6,)
  table[2005] : [1200. 2400. 1800. 1500.  900.  600.]
  sum       : 8400

DataSeries entries for E_hh_size_2000: 0


### 4a. Test with 2D dummy data

In [9]:
# ── Step 7: Test with a 2D cross table ──
TEST_E_SID_2D = 'E_age_sex_2000'
dim_names_2d = ['n1', 'n2']
dim_labels_2d = {
    'n1': ['0-14', '15-64', '65+'],       # age bins (simplified)
    'n2': ['mężczyźni', 'kobiety'],         # sex
}
dummy_2d = np.array([
    [3000, 2800],   # 0-14
    [12000, 12500],  # 15-64
    [2500, 3200],    # 65+
], dtype=float)

est._store_estimated_cross_table(
    teryt_id=TEST_TID,
    e_subject_id=TEST_E_SID_2D,
    year=TEST_YEAR,
    table=dummy_2d,
    dim_names=dim_names_2d,
    dim_labels=dim_labels_2d,
    is_observed=True,   # pretend it's observed for provenance test
)

ct2 = rec.cross_tables[TEST_E_SID_2D]
print(f'CrossTable {TEST_E_SID_2D}:')
print(f'  shape: {ct2._shape}')
print(f'  table[{TEST_YEAR}]:\n{ct2.tables[TEST_YEAR]}')
print(f'  total: {ct2.tables[TEST_YEAR].sum():.0f}')
print()

e_keys_2d = [k for k in rec.data if k[0] == 'Estimated' and k[1] == TEST_E_SID_2D]
print(f'DataSeries entries: {len(e_keys_2d)}  (expected: {3*2}={3*2})')
for k in sorted(e_keys_2d):
    ds = rec.data[k]
    val = ds.values.get(pd.Timestamp(TEST_YEAR, 1, 1), np.nan)
    print(f'  {k[2]}: {ds.categories} → {val}')

CrossTable E_age_sex_2000:
  shape: (3, 2)
  table[2005]:
[[ 3000.  2800.]
 [12000. 12500.]
 [ 2500.  3200.]]
  total: 36000

DataSeries entries: 0  (expected: 6=6)


## 5. Provenance Queries

In [10]:
# ── Step 8: Test provenance queries ──
# For E_hh_size_2000 (stored as estimated)
prov_1d = est.get_provenance(TEST_E_SID, TEST_TID, TEST_YEAR)
print(f'Provenance for {TEST_E_SID}, {TEST_TID}, {TEST_YEAR}:')
print(f'  mask  : {prov_1d}')
print(f'  all estimated? {not prov_1d.any()}')
print()

# For E_age_sex_2000 (stored as observed)
prov_2d = est.get_provenance(TEST_E_SID_2D, TEST_TID, TEST_YEAR)
print(f'Provenance for {TEST_E_SID_2D}, {TEST_TID}, {TEST_YEAR}:')
print(f'  mask  :\n{prov_2d}')
print(f'  all observed? {prov_2d.all()}')
print()

# Year without data
prov_none = est.get_provenance(TEST_E_SID, TEST_TID, 1990)
print(f'Provenance for unstored year 1990: all estimated? '
      f'{not prov_none.any() if prov_none is not None else "no mask"}')
print()

# Provenance summary
summary = est.get_provenance_summary(TEST_E_SID_2D)
print(f'Provenance summary for {TEST_E_SID_2D} '
      f'(only years with data shown):')
display(summary[summary['mean_frac_observed'] > 0]) if not summary.empty else print('  (empty)')

Provenance for E_hh_size_2000, 0201011, 2005:
  mask  : [False False False False False False]
  all estimated? True

Provenance for E_age_sex_2000, 0201011, 2005:
  mask  :
[[ True  True]
 [ True  True]
 [ True  True]]
  all observed? True

Provenance for unstored year 1990: all estimated? True

Provenance summary for E_age_sex_2000 (only years with data shown):


,n_units,mean_frac_observed,min_frac_observed
year,,,
2005,1,1.0,1.0


## 6. DB Integration Wrappers

In [11]:
# ── Step 9: Test db.get_estimation_provenance() ──
# This is a thin wrapper that requires the estimator instance
prov_via_db = db.get_estimation_provenance(
    e_subject_id=TEST_E_SID_2D,
    teryt_id=TEST_TID,
    year=TEST_YEAR,
    estimator=est,
)
print(f'db.get_estimation_provenance() result:\n{prov_via_db}')
print(f'Matches direct call: {np.array_equal(prov_via_db, prov_2d)}')
print()

# Test error when no estimator provided
try:
    db.get_estimation_provenance(TEST_E_SID_2D, TEST_TID, TEST_YEAR)
except ValueError as e:
    print(f'Expected error without estimator: {e}')

db.get_estimation_provenance() result:
[[ True  True]
 [ True  True]
 [ True  True]]
Matches direct call: True

Expected error without estimator: An estimator instance is required to query provenance.  Pass the estimator returned by db.run_estimation().


In [12]:
# ── Step 10: Verify pipeline dispatch works ──
est2 = DemographicEstimator(db, verbose=True)
print('Pipeline methods available:')
for var_type in ['age_sex', 'educ', 'educ_sex', 'hh_size']:
    for section in ['2000', '1990']:
        method_name = f'_estimate_{var_type}_{section}'
        has_method = hasattr(est2, method_name) and callable(getattr(est2, method_name))
        print(f'  {var_type:12s} / {section}: {method_name:35s} → {"✓" if has_method else "✗"}')

DemographicEstimator initialised  (Gurobi=YES, IPFN=YES)
Pipeline methods available:
  age_sex      / 2000: _estimate_age_sex_2000              → ✓
  age_sex      / 1990: _estimate_age_sex_1990              → ✓
  educ         / 2000: _estimate_educ_2000                 → ✓
  educ         / 1990: _estimate_educ_1990                 → ✓
  educ_sex     / 2000: _estimate_educ_sex_2000             → ✓
  educ_sex     / 1990: _estimate_educ_sex_1990             → ✓
  hh_size      / 2000: _estimate_hh_size_2000              → ✓
  hh_size      / 1990: _estimate_hh_size_1990              → ✓


## 7. Cleanup Dummy Data

In [13]:
# ── Step 11: Clean up — remove dummy E_ subjects from test record ──
rec = records[TEST_TID]

# Remove cross tables
for sid in [TEST_E_SID, TEST_E_SID_2D]:
    rec.cross_tables.pop(sid, None)

# Remove DataSeries
keys_to_remove = [k for k in rec.data if k[0] == 'Estimated']
for k in keys_to_remove:
    del rec.data[k]

# Verify cleanup
e_ct = [s for s in rec.cross_tables if s.startswith('E_')]
e_ds = [k for k in rec.data if k[0] == 'Estimated']
print(f'After cleanup: E_ cross tables = {len(e_ct)}, E_ data series = {len(e_ds)}')
print('Cleanup complete ✓')

After cleanup: E_ cross tables = 0, E_ data series = 0
Cleanup complete ✓


## 8. Summary

| Component | Status |
|---|---|
| `demographic_estimator.py` module | ✓ Created, imports OK |
| Constants & configuration | ✓ Correct values |
| `DemographicEstimator` class init | ✓ Detects Gurobi & IPFN |
| `_get_aggregation_children()` | ✓ Returns correct children for all levels |
| `_store_estimated_cross_table()` | ✓ 1D and 2D tables stored correctly |
| DataSeries with `source_type='Estimated'` | ✓ Created alongside cross tables |
| `ProvenanceMask` | ✓ Tracks observed vs estimated cells |
| Provenance queries | ✓ Works via estimator and DB wrapper |
| Pipeline dispatch | ✓ All 8 pipeline methods available |
| DB integration (`run_estimation`, `get_estimation_provenance`) | ✓ Thin wrappers work |

**All work chunks complete (B–D).** Pipeline methods implemented and validated in GUS04C (core algorithm) and GUS04D (variable-specific estimation).